# 01 - Camada Bronze | Ingestão dos Dados Brutos
## Projeto: Pipeline de Zap Imóveis 
### Objetivo: Ler o CSV bruto e salvar como Delta Table na camada Bronze
###AUTOR: Eduardo Cosme Pereira dos Santos 
 

In [0]:
## Importar pacotes :

import datetime
import pandas as pd
import numpy as np
from pyspark.sql.functions import *


In [0]:
%sql
SHOW VOLUMES IN workspace.zap_imoveis


database,volume_name
zap_imoveis,bronze
zap_imoveis,gold
zap_imoveis,silver


In [0]:
## Configuração do logging - logs do projeto 

import logging

logging.basicConfig(
    level=logging.INFO, filename="workspace/zap_imoveis/bronze.log",
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [0]:


## Criando a variavel de caminho 
path_bronze = "/Volumes/workspace/zap_imoveis/bronze"

## ler o arquivo csv
df_bronze = spark.read.csv( f"{path_bronze}/imoveis.csv",header=True,
  inferSchema=True,
  sep=",")



display(df_bronze.head(10))




preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu
6500,1535,680,3,Cruzeiro,null,null,APARTMENT,3,2.0,555
2500,300,98,3,Milionários (Barreiro),null,null,APARTMENT,2,1.0,198
2800,0,120,3,Ipê,null,null,HOME,2,1.0,100
4700,512,1000,3,Buritis,null,null,APARTMENT,3,2.0,412
4000,630,80,3,Buritis,-19.971905,-43.965046,APARTMENT,2,2.0,2554
3500,350,720,3,Silveira,null,null,APARTMENT,3,3.0,266
3300,550,90,3,Estrela do Oriente,-19.964745,-43.984511,APARTMENT,2,2.0,290
6000,0,180,3,Paquetá,null,null,HOME,2,2.0,155
3000,380,116,3,Caiçara-Adelaide,null,null,APARTMENT,2,2.0,190
1950,null,null,3,Conjunto Paulo VI,-19.83209,-43.88831,HOME,2,2.0,0


In [0]:
## criando as variaveis numero de linhas e colunas 
linha_zap = df_bronze.count()
coluna_zap = len(df_bronze.columns)
nulos = df_bronze.select([count(when(col(c).isNull(), c)).alias(c) for c in df_bronze.columns])

print(f"Total de linhas: {linha_zap}")
print(f"Total de colunas: {coluna_zap}")

display(nulos.head(10))



Total de linhas: 1050
Total de colunas: 11


preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu
0,30,61,0,0,579,579,0,0,16,49


In [0]:
## Calcular a porcentagem de dados nulos por coluna com sua porcentagem 
dados_nulos_percentual = nulos.select([round((col(c) / df_bronze.count()) * 100, 2).alias(c) for c in nulos.columns])
display(dados_nulos_percentual)

preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu
0.0,2.86,5.81,0.0,0.0,55.14,55.14,0.0,0.0,1.52,4.67


In [0]:
## importar pacote de atualização de data
from pyspark.sql.functions import current_timestamp
df_bronze = df_bronze.withColumn("data_atualizacao", current_timestamp())
display(df_bronze.head(10))



preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu,data_atualizacao
6500,1535,680,3,Cruzeiro,null,null,APARTMENT,3,2.0,555,2026-05-12T22:44:49.921Z
2500,300,98,3,Milionários (Barreiro),null,null,APARTMENT,2,1.0,198,2026-05-12T22:44:49.921Z
2800,0,120,3,Ipê,null,null,HOME,2,1.0,100,2026-05-12T22:44:49.921Z
4700,512,1000,3,Buritis,null,null,APARTMENT,3,2.0,412,2026-05-12T22:44:49.921Z
4000,630,80,3,Buritis,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:44:49.921Z
3500,350,720,3,Silveira,null,null,APARTMENT,3,3.0,266,2026-05-12T22:44:49.921Z
3300,550,90,3,Estrela do Oriente,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:44:49.921Z
6000,0,180,3,Paquetá,null,null,HOME,2,2.0,155,2026-05-12T22:44:49.921Z
3000,380,116,3,Caiçara-Adelaide,null,null,APARTMENT,2,2.0,190,2026-05-12T22:44:49.921Z
1950,null,null,3,Conjunto Paulo VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:44:49.921Z


In [0]:
##verificar o total de lnhas e colunas
linha_bronze = df_bronze.count()
coluna_bronze = len(df_bronze.columns)
print(f"Total de linhas: {linha_bronze}")
print(f"Total de colunas: {coluna_bronze}")

Total de linhas: 1050
Total de colunas: 12


In [0]:
## Salvar a tabela em formato Delta na camada bronze

df_bronze.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"{path_bronze}/zap_imoveis")



logging.info("tabela delta criada com sucesso na camada bronze")


In [0]:
## Salvar como metrica de qualidade na camada bronze
dados_nulos_percentual.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"{path_bronze}/metricas_bronze")

logging.info("tabela de metricas de qualidade criada com sucesso")

display(dados_nulos_percentual)

preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu
0.0,2.86,5.81,0.0,0.0,55.14,55.14,0.0,0.0,1.52,4.67


In [0]:
## ler os dados da tabela delta
df_bronze = spark.read.format("delta").load(f"{path_bronze}/zap_imoveis")
display(df_bronze.head(10))
## verificar total de linhas e colunas 

print(f"Total de linhas: {df_bronze.count()}")
print(f"Total de colunas: {len(df_bronze.columns)}")


preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu,data_atualizacao
6500,1535,680,3,Cruzeiro,null,null,APARTMENT,3,2.0,555,2026-05-12T22:52:01.415Z
2500,300,98,3,Milionários (Barreiro),null,null,APARTMENT,2,1.0,198,2026-05-12T22:52:01.415Z
2800,0,120,3,Ipê,null,null,HOME,2,1.0,100,2026-05-12T22:52:01.415Z
4700,512,1000,3,Buritis,null,null,APARTMENT,3,2.0,412,2026-05-12T22:52:01.415Z
4000,630,80,3,Buritis,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:52:01.415Z
3500,350,720,3,Silveira,null,null,APARTMENT,3,3.0,266,2026-05-12T22:52:01.415Z
3300,550,90,3,Estrela do Oriente,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:52:01.415Z
6000,0,180,3,Paquetá,null,null,HOME,2,2.0,155,2026-05-12T22:52:01.415Z
3000,380,116,3,Caiçara-Adelaide,null,null,APARTMENT,2,2.0,190,2026-05-12T22:52:01.415Z
1950,null,null,3,Conjunto Paulo VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:52:01.415Z


Total de linhas: 1050
Total de colunas: 12
